# Promoter And Intron Pretrained One-Shot Evaluation

This notebook evaluates the best available `src/learn` promoter and intron checkpoints on the MattLee Lib1 in-house promoter and intron tables.

Scope for this notebook:
- Promoter: score the best DeBoer-core promoter checkpoint on `MattLee_lib1/single_part_variant_level/promoters`.
- Intron: score the best Seelig A5SS checkpoint on `MattLee_lib1/single_part_variant_level/introns`.
- Report Pearson R, Spearman, COD R2, RMSE, and MAE by barcode-quality slice.
- Write generated outputs under ignored `src/finetune/learning_curve/` state.

3'UTR is intentionally out of scope here because its in-house length mismatch needs a separate padding-policy sensitivity analysis.

## Setup

Run this notebook with the `boda_env` kernel. The helper script keeps model loading, sequence transforms, and metric writing reusable outside the notebook.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

repo = Path.cwd().resolve()
while repo.name != "boda2_EU" and repo.parent != repo:
    repo = repo.parent
if repo.name != "boda2_EU":
    raise RuntimeError("Could not locate boda2_EU repo root from current working directory")

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.finetune.finetune_sweep_scripts import promoter_intron_inhouse_one_shot_eval as eval_promoter_intron

OUTDIR = repo / "src" / "finetune" / "learning_curve" / "promoter_intron_pretrained_inhouse_eval_may2026"
OUTDIR

## Candidate Models And Transform Policies

The runner selects the highest public validation metric from `src/learn/run_registry/runs.csv` with an existing local tarball artifact.

- Promoter uses right `N` padding to the checkpoint input length, matching `PromoterDataModule` behavior.
- Intron uses center `N` padding to the Seelig checkpoint input length. This is only a diagnostic transfer check because the Seelig model predicts splice-site usage, while the in-house target is RNA/DNA.

In [ ]:
candidate_rows = []
for spec in eval_promoter_intron.DEFAULT_SPECS:
    row = eval_promoter_intron.select_best_run(
        eval_promoter_intron.DEFAULT_RUNS_CSV,
        task_family=spec.task_family,
        target_family=spec.target_family,
    )
    candidate_rows.append({
        "part_type": spec.part_type,
        "run_id": row["run_id"],
        "task_family": row["task_family"],
        "target_family": row["target_family"],
        "model_module": row["model_module"],
        "public_metric_name": row["best_metric_name"],
        "public_metric_value": row["best_metric_value"],
        "sequence_column": spec.sequence_column,
        "transform_policy": spec.transform_policy,
        "artifact_path": row["artifact_path"],
    })

pd.DataFrame(candidate_rows)

## Run One-Shot Evaluation

This writes per-part audits, predictions, metrics, and model cards under `OUTDIR`. The output directory is generated state and should not be committed unless a small summary is deliberately curated.

In [ ]:
result = eval_promoter_intron.run_default_evaluations(
    outdir=OUTDIR,
    device="cuda" if eval_promoter_intron.torch.cuda.is_available() else "cpu",
    batch_size=512,
)

result["combined_metrics_path"], result["manifest_path"]

## Primary Diagnostic Metrics

For one-shot transfer into in-house RNA/DNA, Pearson R and Spearman are the primary readouts. COD R2 is retained as a calibration/scale warning, but these public-pretrained outputs are not expected to be calibrated to in-house RNA/DNA without finetuning.

In [ ]:
metrics = pd.read_csv(result["combined_metrics_path"])
primary = metrics[
    (metrics["target_column"] == "log2_RNA_DNA")
    & (metrics["prediction_column"] == "pred_output_0")
    & (metrics["subset_id"].isin(["all", "bc_ge_4", "bc_ge_8", "bc_ge_16"]))
].copy()

primary[[
    "part_type", "run_id", "subset_id", "n",
    "pearson", "spearman", "cod_r2", "rmse", "mae"
]].sort_values(["part_type", "subset_id"])

## Data Audits

Check these before interpreting the metrics. They record length distributions, invalid sequence counts, barcode-count ranges, and the transform policy used to make each in-house part compatible with the pretrained checkpoint input length.

In [ ]:
import json

audit_rows = []
for part in result["manifest"]["parts"]:
    audit = part["audit"]
    audit_rows.append({
        "part_type": audit["part_type"],
        "run_id": audit["run_row"]["run_id"],
        "n_raw_rows": audit["n_raw_rows"],
        "n_usable_rows": audit["n_usable_rows"],
        "n_invalid_original": audit["n_invalid_original"],
        "model_input_len": audit["model_input_len"],
        "transform_policy": audit["transform_policy"],
        "barcode_median": audit["barcode_count_summary"]["median"],
        "barcode_max": audit["barcode_count_summary"]["max"],
        "audit_path": part["paths"]["audit_path"],
    })

pd.DataFrame(audit_rows)

## Interpretation Notes

- Promoter one-shot scores should be interpreted as a weak ranking diagnostic unless they are stable across barcode-quality slices.
- Intron one-shot scores are especially cautious because the source task is Seelig A5SS splice-site usage, not construct RNA/DNA. A negative or unstable correlation should point toward intron-specific finetuning or a from-scratch in-house intron baseline rather than direct promotion.
- Generated prediction files can be used later by the part-model promotion scaffold, but only small curated summaries should be committed.